# 직접 해보기 02 · 달의 두 얼굴, 지각 두께로 비교하기

GRAIL 중력 자료로 계산한 **모형**의 지각 두께(Wieczorek 외 2013, Zenodo 10.5281/zenodo.997347)를 읽는다. 사진도 직접 관측값도 아니다.

같은 계산이 `analysis.py`에도 들어 있다.

In [ ]:
import analysis as A
rows = A.load()
print(A.sha8(), len(rows), '행')
print(rows[:2])

## 1. 모형 1의 지각 두께 지도를 그려 본다
matplotlib이 없으면 `pip install matplotlib`. 점선 사이(경도 -90~+90)가 앞면이다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
grid = np.full((90, 180), np.nan)
for la, lo, t1, t3 in rows:
    grid[int((90 - la) // 2), int((lo + 180) // 2)] = t1
plt.figure(figsize=(9, 4.5))
plt.imshow(grid, extent=(-180, 180, -90, 90), cmap='YlOrBr')
plt.colorbar(label='km')
plt.axvline(-90, ls='--', c='k'); plt.axvline(90, ls='--', c='k')
plt.xlabel('경도'); plt.ylabel('위도'); plt.show()

## 2. 앞면과 뒷면의 평균 (책의 예상 결과)
각 칸에 cos(위도)를 곱해 평균한다.

In [ ]:
for m in (1, 3):
    r = A.hemisphere_means(rows, model=m)
    print(f"모형 {m}: 앞 {r['near']:.2f} · 뒤 {r['far']:.2f} · 차 {r['diff']:.2f} km")

## 3. 한 번에 하나만 바꿔 본다
위도 범위를 좁히고, 넓이 가중치를 빼 본다. 바꾼 조건과 결과를 연구 노트에 적어 두자.

In [ ]:
for lat in (90, 60, 30, 15):
    for w in (True, False):
        d = [A.hemisphere_means(rows, m, lat, w)['diff'] for m in (1, 3)]
        print(f'위도 ±{lat:>2}° 가중={w!s:<5}', [round(x, 2) for x in d])

## 4. 평균이 가린 것
모형 1에서 뒷면 넓이 중 앞면 평균보다 얇은 부분의 비율.

In [ ]:
print(round(A.thin_far_share(rows, 1) * 100, 1), '%')